# total.ipynb

- 목적: 통합 대피소 원본 CSV를 앱에서 쓰기 쉬운 컬럼 구조로 정리하고 프로젝트 대상 권역만 남긴다.
- 입력: `integrated_shelter_raw.csv`
- 출력: `project_3_pr_total_shelter_clean.csv`, `project_3_pre_shelter_final.csv`
- 메모: 통합 대피소의 기본 컬럼명 정리와 주소 기반 권역 추출을 담당하는 노트북이다.


## 1. 통합 대피소 원본 로딩
원본 CSV를 읽고 어떤 컬럼이 들어 있는지 먼저 확인한다.


In [1]:
# 통합 대피소 원본을 불러와 데이터 크기와 컬럼 구조를 먼저 확인한다.
import pandas as pd

df = pd.read_csv("data/integrated_shelter_raw.csv")

print("원본 데이터 수:", len(df))
print(df.columns)


FileNotFoundError: [Errno 2] No such file or directory: 'data/integrated_shelter_raw.csv'

## 원본 컬럼 메모
- `REARE_NM`: 대피소 이름
- `LOT`: 경도
- `MNG_SN`: 관리 번호
- `RONA_DADDR`: 도로명 주소
- `SHLT_SE_NM`: 대피소 유형
- `LAT`: 위도
- `SHLT_SE_CD`: 대피소 유형 코드


In [ ]:
df.head()


## 2. 컬럼명 표준화와 기본 정리
앱에서 바로 쓸 수 있도록 한글 컬럼명으로 바꾸고 좌표/중복을 정리한다.


In [ ]:
df = df.rename(columns={
    "REARE_NM": "대피소명",
    "LOT": "경도",
    "LAT": "위도",
    "RONA_DADDR": "주소",
    "SHLT_SE_NM": "대피소유형",
    "SHLT_SE_CD": "대피소코드",
    "MNG_SN": "관리번호"
})


In [ ]:
df = df[
    [
        "대피소명",
        "주소",
        "대피소유형",
        "위도",
        "경도"
    ]
]


In [ ]:
df = df.dropna(subset=["위도", "경도"])


In [ ]:
df = df.drop_duplicates(
    subset=["대피소명", "주소"]
)


In [ ]:
df


## 3. 주소 분해와 지역 컬럼 생성
주소 문자열에서 시도·시군구를 뽑아 지역 필터링 기준을 만든다.


In [ ]:
# 주소 문자열에서 시도/시군구/구를 분리하기 위한 함수다.
def clean_korea_address(address):

    if pd.isna(address):
        return None, None, None

    parts = address.split()

    sido_map = {
        "서울특별시": "서울",
        "부산광역시": "부산",
        "대구광역시": "대구",
        "인천광역시": "인천",
        "광주광역시": "광주",
        "대전광역시": "대전",
        "울산광역시": "울산",
        "세종특별자치시": "세종",

        "경기도": "경기",
        "강원도": "강원",
        "강원특별자치도": "강원",

        "충청북도": "충북",
        "충청남도": "충남",

        "전라북도": "전북",
        "전북특별자치도": "전북",

        "전라남도": "전남",

        "경상북도": "경북",
        "경상남도": "경남",

        "제주특별자치도": "제주"
    }

    sido = parts[0] if len(parts) > 0 else None
    sigungu = parts[1] if len(parts) > 1 else None
    gu = parts[2] if len(parts) > 2 else None

    if sido in sido_map:
        sido = sido_map[sido]

    return sido, sigungu, gu


In [ ]:
df[["시도", "시군구", "구"]] = (
    df["주소"]
    .apply(lambda x: pd.Series(clean_korea_address(x)))
)


In [ ]:
df[["주소", "시도", "시군구", "구"]].head()


In [ ]:
df["지역"] = (
    df["시도"] + " " +
    df["시군구"]
)


In [ ]:
df.to_csv(
    "data/project_3_pr_total_shelter_clean.csv",
    index=False,
    encoding="utf-8-sig"
)


## 4. 프로젝트 대상 권역 필터링과 저장
동남권 대상 지역만 남겨 후속 병합용 통합 대피소 CSV를 만든다.


In [ ]:
target_region = ["울산", "경북", "경남", "대구", "부산"]


In [ ]:
df = df[df["시도"].isin(target_region)]


In [ ]:
df["시도"].value_counts()


In [ ]:
len(df)


In [ ]:
df.to_csv(
    "data/project_3_pre_shelter_final.csv",
    index=False,
    encoding="utf-8-sig"
)
